# Road Damage Detection - YOLOv11 Medium Training (T4 GPU)

Trains `yolo11m` on the road damage dataset from Google Drive.
Optimized to saturate a Colab T4 (16 GB VRAM).

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload the dataset folder `road damage det.v4i.yolov11 (1)` to your Google Drive (e.g. `MyDrive/datasets/`)
3. Update `DRIVE_DATASET_PATH` below if your path differs

## 1. Verify GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime → Change runtime type → T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 2. Install Ultralytics

In [ ]:
# Clean install — uninstall first to avoid mixed-version state from any previous
# partial installs, then install the latest ultralytics.
# If this is a fresh runtime you can skip the uninstall; it's a no-op.
!pip uninstall -y ultralytics 2>/dev/null
!pip install -q -U ultralytics

import ultralytics
ultralytics.checks()

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Copy dataset to local SSD (parallel multi-threaded)

Drive's bottleneck on small files is per-file latency, not bandwidth. `shutil.copytree` copies one file at a time, which is painfully slow.
**Fix:** copy many files in parallel with a `ThreadPoolExecutor` (Drive handles ~32 parallel reads well).

In [ ]:
import os, shutil, yaml, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# EDIT THIS if your Drive path is different
DRIVE_DATASET_PATH = '/content/drive/MyDrive/datasets/road damage det.v4i.yolov11 (1)'
LOCAL_DATASET_PATH = '/content/dataset'
COPY_THREADS = 32   # Drive handles ~32 parallel reads well

assert os.path.exists(DRIVE_DATASET_PATH), f'Not found: {DRIVE_DATASET_PATH}'

if not os.path.exists(LOCAL_DATASET_PATH):
    src_root = Path(DRIVE_DATASET_PATH)
    dst_root = Path(LOCAL_DATASET_PATH)

    # Build the (src, dst) task list and pre-create directories
    tasks = []
    for src in src_root.rglob('*'):
        if src.is_file():
            rel = src.relative_to(src_root)
            dst = dst_root / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            tasks.append((src, dst))

    print(f'Copying {len(tasks)} files with {COPY_THREADS} parallel threads ...')
    t0 = time.time()

    def copy_one(src_dst):
        src, dst = src_dst
        if not dst.exists():
            shutil.copy2(src, dst)

    with ThreadPoolExecutor(max_workers=COPY_THREADS) as ex:
        futures = [ex.submit(copy_one, t) for t in tasks]
        for _ in tqdm(as_completed(futures), total=len(futures), desc='copying'):
            pass

    print(f'Done in {time.time()-t0:.1f}s')
else:
    print('Local dataset already present.')

# Rewrite data.yaml with absolute paths
yaml_path = os.path.join(LOCAL_DATASET_PATH, 'data.yaml')
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)
cfg['train'] = os.path.join(LOCAL_DATASET_PATH, 'train/images')
cfg['val']   = os.path.join(LOCAL_DATASET_PATH, 'valid/images')
cfg['test']  = os.path.join(LOCAL_DATASET_PATH, 'test/images')
with open(yaml_path, 'w') as f:
    yaml.safe_dump(cfg, f)
print(cfg)

### 4c. Verify all classes are present
Scans label files to confirm every class in `data.yaml` actually appears in the dataset (and prints per-split counts). If any class has 0 instances, training will silently ignore it.

In [ ]:
from pathlib import Path
from collections import Counter
import yaml

with open(f'{LOCAL_DATASET_PATH}/data.yaml') as f:
    cfg = yaml.safe_load(f)
class_names = cfg['names']
nc = cfg['nc']
print(f'data.yaml declares {nc} classes: {class_names}\
')

split_dirs = {
    'train': Path(LOCAL_DATASET_PATH) / 'train' / 'labels',
    'valid': Path(LOCAL_DATASET_PATH) / 'valid' / 'labels',
    'test':  Path(LOCAL_DATASET_PATH) / 'test'  / 'labels',
}

total = Counter()
print(f\"{'class':<25}{'train':>10}{'valid':>10}{'test':>10}{'total':>10}\")
print('-' * 65)

per_split = {s: Counter() for s in split_dirs}
for split, lbl_dir in split_dirs.items():
    if not lbl_dir.exists():
        continue
    for lbl_file in lbl_dir.glob('*.txt'):
        for line in lbl_file.read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            cid = int(line.split()[0])
            per_split[split][cid] += 1
            total[cid] += 1

for cid, name in enumerate(class_names):
    row = f\"{cid}: {name:<22}\"
    for split in ('train', 'valid', 'test'):
        row += f\"{per_split[split].get(cid, 0):>10}\"
    row += f\"{total.get(cid, 0):>10}\"
    print(row)

missing = [class_names[i] for i in range(nc) if total.get(i, 0) == 0]
if missing:
    print(f'\
 WARNING: these classes have ZERO instances and will not be learned: {missing}')
else:
    print(f'\
 All {nc} classes have instances in the dataset.')

## 5. Train YOLOv11m (T4-optimized)

Key flags for maximum T4 utilization:
- `batch=-1` → AutoBatch picks the largest batch that fits in 16 GB VRAM
- `amp=True` → mixed precision (T4 supports FP16 → ~2× throughput)
- `cache='ram'` → keeps images in RAM (Colab has ~12 GB, usually enough for this dataset)
- `workers=8` → parallel data loading
- `imgsz=640` → standard; bump to 800 only if VRAM allows

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11m.pt')

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=-1,              # AutoBatch: max that fits on T4
    cache='ram',           # cache images in RAM
    workers=8,
    device=0,
    amp=True,              # FP16 mixed precision
    optimizer='SGD',
    lr0=0.01,
    cos_lr=True,           # cosine LR schedule
    close_mosaic=10,       # disable mosaic last 10 epochs
    patience=20,           # early stop if no improvement
    project='/content/runs',
    name='road_damage_yolo11m',
    exist_ok=True,
    plots=True,
    seed=42,
)

## 6. Validate on test set

In [ ]:
best = '/content/runs/road_damage_yolo11m/weights/best.pt'
model = YOLO(best)
metrics = model.val(data='/content/dataset/data.yaml', split='test', imgsz=640)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

## 7. Save results back to Google Drive

In [ ]:
import shutil
DRIVE_OUTPUT = '/content/drive/MyDrive/road_damage_yolo11m_run'
if os.path.exists(DRIVE_OUTPUT):
    shutil.rmtree(DRIVE_OUTPUT)
shutil.copytree('/content/runs/road_damage_yolo11m', DRIVE_OUTPUT)
print('Saved to:', DRIVE_OUTPUT)

## 8. (Optional) Quick inference test

In [ ]:
from ultralytics import YOLO
from PIL import Image
import glob

model = YOLO('/content/runs/road_damage_yolo11m/weights/best.pt')
sample = glob.glob('/content/dataset/test/images/*.jpg')[0]
res = model.predict(sample, imgsz=640, conf=0.25, save=True)
print('Saved prediction to:', res[0].save_dir)